# Генерация для бота: Transformer 107M / контекст 256

Выберите **NVIDIA T4** в Colab или Kaggle. Ноутбук скачивает официальный
`test` из датасета по ссылке в задании: 1919 вопросов без эталонных ответов.
Файл `pass1_questions.jsonl` создаётся в исходном порядке строк этого теста.
Одна программа на вопрос, жадная генерация, максимум **256 новых токенов**.
Последняя ячейка измеряет загрузку модели, токенизацию, генерацию и запись.
Установка зависимостей и скачивание файлов выполняются заранее.
Выполнение на H100 не подтверждает ограничение 180 секунд на T4.


In [ ]:
%pip -q install hydra-core==1.3.2 tokenizers==0.23.3 pyarrow==23.0.1


In [ ]:
from pathlib import Path
import base64, io, json, sys, zipfile
work = Path.cwd()
root = work / 'code107m_inference'
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    archive.extractall(root)
sys.path.insert(0, str(root))
questions = work / 'pass1_questions.jsonl'
from scripts.prepare_questions import prepare_questions
if not questions.is_file():
    print(prepare_questions(questions))
assert len(questions.read_text(encoding='utf-8').splitlines()) == 1919


## Веса текущего запуска

Можно заранее распаковать `hw2_submission.zip` в рабочий каталог.
Следующая ячейка при отсутствии весов скачает части архива из репозитория,
проверит SHA-256 каждой части и всего архива и распакует его.
Иные чекпойнты не загружаются.


In [ ]:
import concurrent.futures
import urllib.request
from scripts.assemble_checkpoint import assemble, extract
model_dir = work / 'model'
if not (model_dir / 'best.pt').is_file():
    model_dir = work / 'weights_bundle' / 'model'
if not (model_dir / 'best.pt').is_file():
    base = 'https://raw.githubusercontent.com/Kera-0/-/main/delivery/'
    parts_dir = work / 'delivery'
    parts_dir.mkdir(exist_ok=True)
    def download(name):
        assert Path(name).name == name
        target = parts_dir / name
        if not target.is_file():
            with urllib.request.urlopen(base + name, timeout=180) as response:
                temporary = target.with_suffix(target.suffix + '.part')
                with temporary.open('wb') as handle:
                    while block := response.read(1024 * 1024):
                        handle.write(block)
                temporary.replace(target)
        return target
    manifest_path = download('manifest.json')
    manifest = json.loads(manifest_path.read_text())
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(download, [p['name'] for p in manifest['parts']]))
    archive = assemble(manifest_path, work / 'hw2_submission.zip')
    extract(archive, model_dir.parent)
assert (model_dir / 'tokenizer.json').is_file()


In [ ]:
from argparse import Namespace
from inference import run
report = run(Namespace(
    checkpoint=str(model_dir / 'best.pt'),
    tokenizer=str(model_dir / 'tokenizer.json'),
    questions=str(questions), output=str(work / 'submission_256.jsonl'),
    batch_size=128, max_new_tokens=256, device='cuda', dtype='fp16',
    expected_count=1919, require_t4=True, expected_parameters=106977792,
))
from scripts.validate_submission import validate
validation = validate(questions, work / 'submission_256.jsonl', work / 'submission_256.details.jsonl')
assert validation['order_verified']
print(validation)
assert report['under_180_seconds'], f"Превышен лимит T4: {report['total_seconds']:.2f} с"
print('Готово: отправьте submission_256.jsonl боту для вычисления pass@1')
